# SpaceX API snapshot and enriched course data

This notebook inspects the archived API response supplied with the course and validates the saved, enriched Falcon 9 feature table. It does not claim to repeat online enrichment calls. The original API is `https://api.spacexdata.com/v4/launches/past`. IBM supplied the archived JSON and enriched CSV.

In [1]:
from pathlib import Path
import os
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
os.chdir(root)
assert (root / 'data').is_dir(), 'Run from the repository root or notebooks directory.'

import pandas as pd
import json
raw = json.loads(Path('data/API_call_spacex_api.json').read_text())
launches = pd.json_normalize(raw)
print('Archived API launches:', len(launches))
display(launches[['flight_number','name','date_utc','rocket','launchpad']].head())

Archived API launches: 107


,flight_number,name,date_utc,rocket,launchpad
0,1,FalconSat,2006-03-24T22:30:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
1,2,DemoSat,2007-03-21T01:10:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
2,3,Trailblazer,2008-08-03T03:34:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
3,4,RatSat,2008-09-28T23:15:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86
4,5,RazakSat,2009-07-13T03:35:00.000Z,5e9d0d95eda69955f709d1eb,5e9e4502f5090995de566f86


## Scope filters
Keep single-core, single-payload records through 13 November 2020. Resolve rocket, launchpad, payload and core IDs through their API endpoints in an online collection. Here, the course table supplies those enriched fields.

In [2]:
selected = launches[launches.cores.map(len).eq(1) & launches.payloads.map(len).eq(1)].copy()
selected['Date'] = pd.to_datetime(selected.date_utc, utc=True).dt.strftime('%Y-%m-%d')
selected = selected[selected.Date <= '2020-11-13']
print('Single-core / single-payload records:', len(selected))
display(selected[['flight_number','name','Date','rocket']].head())

Single-core / single-payload records: 94


,flight_number,name,Date,rocket
0,1,FalconSat,2006-03-24,5e9d0d95eda69955f709d1eb
1,2,DemoSat,2007-03-21,5e9d0d95eda69955f709d1eb
3,4,RatSat,2008-09-28,5e9d0d95eda69955f709d1eb
4,5,RazakSat,2009-07-13,5e9d0d95eda69955f709d1eb
5,6,Falcon 9 Test Flight,2010-06-04,5e9d0d95eda69973a809d1ec


In [3]:
features = pd.read_csv('data/dataset_part_1.csv')
assert features.shape == (90,17)
assert features.BoosterVersion.eq('Falcon 9').all()
assert features.FlightNumber.tolist() == list(range(1,91))
assert set(features.Date).issubset(set(selected.Date))
print('Enriched Falcon 9 sample:', features.shape)
print('Dates:', features.Date.min(), 'to', features.Date.max())
display(features.head())
display(features.isna().sum().to_frame('Missing values'))

Enriched Falcon 9 sample: (90, 17)
Dates: 2010-06-04 to 2020-11-05


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


,Missing values
FlightNumber,0
Date,0
BoosterVersion,0
PayloadMass,0
Orbit,0
LaunchSite,0
Outcome,0
Flights,0
GridFins,0
Reused,0


Payload mass is already mean-imputed in the supplied table. LandingPad has missing values because many early launches had no assigned landing pad. The saved table is the analytical source, and matching dates checks coverage without reconstructing missing enrichment responses.